In [1]:
# =============================================================================
# 05_G-1.ipynb — 셀 1: 환경 / 경로 / 유틸
# =============================================================================
# 목적: G팩터 (Short-term Reversal) 검증
# 가설: 단기 급락 종목이 평균회귀로 반등
# 논리: 과잉반응 → 복원, 유동성 충격 → 해소
# 선행연구: Jegadeesh (1990), Lehmann (1990)
# =============================================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams["figure.figsize"] = (14, 5)
plt.rcParams["axes.grid"] = True

# ── SSOT 경로 ─────────────────────────────────────────────────────
QP2_ROOT = Path(r"C:\QP2")
DATA_DIR = QP2_ROOT / "data"
INTERIM_DIR = DATA_DIR / "interim"
META_DIR = DATA_DIR / "meta"

PATHS = {
    "px_wide":   INTERIM_DIR / "yahoo_adjclose_wide.parquet",
    "regime":    INTERIM_DIR / "regime_v4.parquet",
    "universe":  META_DIR / "sp500_universe.parquet",
}

# ── G-1 전용 파라미터 ─────────────────────────────────────────────
TOP_N = 30              # 포트폴리오 종목 수
COST_BP = 20            # 편도 거래비용 (bp)
REGIME_COL = "regime"

# Reversal 파라미터
REV_LOOKBACK = 5        # 과거 N일 수익률 (formation)
REV_HOLD = 5            # 보유 기간 (일)

# ── 공통 유틸 함수 ────────────────────────────────────────────────
def calc_perf(ret_series):
    """수익률 시리즈 → CAGR, Sharpe, MaxDD"""
    r = ret_series.dropna()
    if len(r) < 20:
        return {"CAGR": np.nan, "Sharpe": np.nan, "MaxDD": np.nan, "N": len(r)}
    cum = (1 + r).cumprod()
    years = len(r) / 252
    cagr = cum.iloc[-1] ** (1 / years) - 1 if years > 0 else np.nan
    sharpe = r.mean() / r.std() * np.sqrt(252) if r.std() > 0 else 0
    dd = cum / cum.cummax() - 1
    maxdd = dd.min()
    return {"CAGR": cagr, "Sharpe": sharpe, "MaxDD": maxdd, "N": len(r)}

def calc_tstat(port_ret, bm_ret):
    """초과수익 t-stat"""
    excess = (port_ret - bm_ret).dropna()
    if len(excess) < 2:
        return np.nan
    return excess.mean() / (excess.std() / np.sqrt(len(excess)))

print("=" * 60)
print("05_G-1.ipynb — G팩터 (Short-term Reversal)")
print("=" * 60)
print(f"QP2_ROOT     : {QP2_ROOT}")
print(f"REV_LOOKBACK : {REV_LOOKBACK}일 (formation)")
print(f"REV_HOLD     : {REV_HOLD}일 (holding)")
print(f"TOP_N        : {TOP_N}")
print(f"REGIME_COL   : {REGIME_COL}")

05_G-1.ipynb — G팩터 (Short-term Reversal)
QP2_ROOT     : C:\QP2
REV_LOOKBACK : 5일 (formation)
REV_HOLD     : 5일 (holding)
TOP_N        : 30
REGIME_COL   : regime


In [2]:
# =============================================================================
# 셀 2 — 데이터 로드
# =============================================================================
# 산출물: px_wide, ret_1d, regime_d, ew_ret_d, universe
# =============================================================================

# ── 주가 (일간) ───────────────────────────────────────────────────
px_wide = pd.read_parquet(PATHS["px_wide"])
if "date" in px_wide.columns:
    px_wide = px_wide.set_index("date")
px_wide.index = pd.to_datetime(px_wide.index)

# ── 유니버스 ──────────────────────────────────────────────────────
universe = pd.read_parquet(PATHS["universe"])
common_tk = sorted(set(universe["ticker_yahoo"]) & set(px_wide.columns))
px_wide = px_wide[common_tk]

# ── 일간 수익률 ───────────────────────────────────────────────────
ret_1d = px_wide.pct_change()

# ── EW 벤치마크 (일간) ────────────────────────────────────────────
ew_ret_d = ret_1d.mean(axis=1)
ew_ret_d.name = "EW"

# ── 레짐 (일간) ───────────────────────────────────────────────────
regime_raw = pd.read_parquet(PATHS["regime"])
if "date" in regime_raw.columns:
    regime_raw = regime_raw.set_index("date")
regime_raw.index = pd.to_datetime(regime_raw.index)

# 일간으로 리샘플 (ffill)
regime_d = regime_raw[[REGIME_COL]].reindex(ret_1d.index, method="ffill")

# ── 분석 기간 설정 (2013-06-19 이후) ──────────────────────────────
START_DATE = "2013-06-19"
px_wide = px_wide.loc[START_DATE:]
ret_1d = ret_1d.loc[START_DATE:]
ew_ret_d = ew_ret_d.loc[START_DATE:]
regime_d = regime_d.loc[START_DATE:]

print(f"px_wide  : {px_wide.shape[0]} days × {px_wide.shape[1]} tickers")
print(f"ret_1d   : {ret_1d.shape}")
print(f"기간     : {ret_1d.index.min().date()} ~ {ret_1d.index.max().date()}")
print(f"레짐 분포:\n{regime_d[REGIME_COL].value_counts().sort_index()}")

px_wide  : 3181 days × 503 tickers
ret_1d   : (3181, 503)
기간     : 2013-06-19 ~ 2026-02-10
레짐 분포:
regime
Bear        486
Bull       2426
Neutral     269
Name: count, dtype: int64


In [3]:
# =============================================================================
# 셀 3 — Reversal 시그널 생성
# =============================================================================
# 목적: 과거 N일 수익률 기준 "급락 종목" 식별
# 산출물: rev_signal (일별 × 종목별 reversal z-score, 급락 = 양수)
# =============================================================================

print("=" * 60)
print("G-1 Reversal 시그널 생성")
print("=" * 60)

# ── 과거 N일 누적 수익률 ──────────────────────────────────────────
past_ret = ret_1d.rolling(window=REV_LOOKBACK).sum()

# ── 횡단면 z-score (급락 = 양수로 변환) ───────────────────────────
rev_signal = pd.DataFrame(index=past_ret.index, columns=past_ret.columns, dtype=float)

for date in past_ret.index:
    row = past_ret.loc[date].dropna()
    if len(row) < 50:
        continue
    z = (row - row.mean()) / row.std()
    rev_signal.loc[date, z.index] = -z  # 부호 반전: 급락 = 양수

print(f"rev_signal shape: {rev_signal.shape}")
print(f"유효 시작: {rev_signal.dropna(how='all').index.min().date()}")
print(f"일평균 유효 종목: {rev_signal.notna().sum(axis=1).mean():.0f}")
print(f"\n분포:\n{rev_signal.stack().describe().round(3)}")

G-1 Reversal 시그널 생성
rev_signal shape: (3181, 503)
유효 시작: 2013-06-25
일평균 유효 종목: 482

분포:
count    1533542.000
mean           0.000
std            0.999
min          -18.823
25%           -0.507
50%            0.008
75%            0.519
max           17.638
dtype: float64


In [4]:
# =============================================================================
# 셀 4 — G-1 백테스트 (전체 기간)
# =============================================================================
# 목적: 급락 종목 매수 → N일 보유 → 수익률 측정
# 산출물: g1_results
# =============================================================================

print("=" * 60)
print("G-1 Reversal 백테스트")
print("=" * 60)

def backtest_reversal(signal_df, ret_df, regime_df, top_n=30, hold_days=5):
    """
    Reversal 백테스트
    - 매일 시그널 상위 N종목 (급락) 매수
    - hold_days 후 수익률 측정
    """
    results = []
    dates = signal_df.index.tolist()
    
    for i, date in enumerate(dates):
        if i + hold_days >= len(dates):
            break
            
        sig = signal_df.loc[date].dropna()
        if len(sig) < top_n:
            continue
        
        # 상위 N종목 (급락 = 시그널 높음)
        top_tickers = sig.nlargest(top_n).index.tolist()
        
        # hold_days 후 날짜
        exit_date = dates[i + hold_days]
        
        # 보유 기간 수익률 (복리)
        hold_ret = (1 + ret_df.loc[dates[i+1]:exit_date, top_tickers]).prod() - 1
        port_ret = hold_ret.mean()
        
        # EW 벤치마크
        ew_hold = (1 + ret_df.loc[dates[i+1]:exit_date].mean(axis=1)).prod() - 1
        
        # 레짐
        reg = regime_df.loc[date, REGIME_COL] if date in regime_df.index else "Unknown"
        
        results.append({
            "entry_date": date,
            "exit_date": exit_date,
            "g1_ret": port_ret,
            "ew_ret": ew_hold,
            "regime": reg,
        })
    
    return pd.DataFrame(results)

# ── 백테스트 실행 ─────────────────────────────────────────────────
g1_results = backtest_reversal(rev_signal, ret_1d, regime_d, top_n=TOP_N, hold_days=REV_HOLD)
g1_results = g1_results.set_index("entry_date")

print(f"총 {len(g1_results)} 거래")
print(f"기간: {g1_results.index.min().date()} ~ {g1_results.index.max().date()}")

# ── 성과 계산 ─────────────────────────────────────────────────────
# 일별 수익률로 변환 (hold_days로 나눔)
g1_daily = g1_results["g1_ret"] / REV_HOLD
ew_daily = g1_results["ew_ret"] / REV_HOLD

g1_perf = calc_perf(g1_daily)
ew_perf = calc_perf(ew_daily)
t_stat = calc_tstat(g1_results["g1_ret"], g1_results["ew_ret"])

print(f"\n{'지표':12s} {'G-1 Rev':>12s} {'EW BM':>12s}")
print("-" * 40)
print(f"{'CAGR':12s} {g1_perf['CAGR']:>12.2%} {ew_perf['CAGR']:>12.2%}")
print(f"{'Sharpe':12s} {g1_perf['Sharpe']:>12.3f} {ew_perf['Sharpe']:>12.3f}")
print(f"{'MaxDD':12s} {g1_perf['MaxDD']:>12.2%} {ew_perf['MaxDD']:>12.2%}")
print(f"{'N trades':12s} {g1_perf['N']:>12d} {ew_perf['N']:>12d}")
print(f"\nt-stat vs EW: {t_stat:.3f}")

# ── 승률 ──────────────────────────────────────────────────────────
win_rate = (g1_results["g1_ret"] > g1_results["ew_ret"]).mean()
print(f"승률 (vs EW): {win_rate:.1%}")

G-1 Reversal 백테스트
총 3172 거래
기간: 2013-06-25 ~ 2026-02-03

지표                G-1 Rev        EW BM
----------------------------------------
CAGR               29.01%       19.05%
Sharpe              2.179        2.340
MaxDD             -47.69%      -33.34%
N trades             3172         3172

t-stat vs EW: 4.498
승률 (vs EW): 52.7%


In [5]:
# =============================================================================
# 셀 4-1 — G-1b: 급등 후 하락 (Reversal 반대편) 검증
# =============================================================================
# 목적: 급등 종목 매수 → N일 보유 → "오히려 빠지는지" 확인
#        즉, 급등 Top30이 EW 대비 언더퍼폼하면 회피 시그널로 활용
# 산출물: g1b_results
# =============================================================================

print("=" * 60)
print("G-1b: 급등 종목 매수 (반전 반대편)")
print("=" * 60)

# ── 급등 시그널: z-score 그대로 (뒤집지 않음) ──────────────────────
# rev_signal은 -z (급락=양수). 여기서는 +z (급등=양수)
surge_signal = -rev_signal  # 부호 반전 → 급등이 상위

# ── 백테스트: 급등 Top30 매수 ──────────────────────────────────────
g1b_results = backtest_reversal(surge_signal, ret_1d, regime_d, top_n=TOP_N, hold_days=REV_HOLD)
g1b_results = g1b_results.set_index("entry_date")

print(f"총 {len(g1b_results)} 거래")

# ── 전체 성과 ──────────────────────────────────────────────────────
t_all = calc_tstat(g1b_results["g1_ret"], g1b_results["ew_ret"])
excess_all = (g1b_results["g1_ret"].mean() - g1b_results["ew_ret"].mean()) * (252 / REV_HOLD)
print(f"\n전체: 초과수익(연율) = {excess_all:+.2%}, t = {t_all:.2f}")

# ── 레짐별 ─────────────────────────────────────────────────────────
print(f"\n{'레짐':22s} {'trades':>7s} {'Surge avg':>10s} {'EW avg':>10s} {'초과':>10s} {'t-stat':>8s} {'승률':>7s}")
print("-" * 85)

for reg in sorted(g1b_results["regime"].unique()):
    sub = g1b_results[g1b_results["regime"] == reg]
    if len(sub) < 20:
        print(f"{reg:22s} {len(sub):>7d}   (샘플 부족)")
        continue
    
    g1b_avg = sub["g1_ret"].mean()
    ew_avg = sub["ew_ret"].mean()
    excess = g1b_avg - ew_avg
    t = calc_tstat(sub["g1_ret"], sub["ew_ret"])
    win = (sub["g1_ret"] > sub["ew_ret"]).mean()
    
    g1b_ann = g1b_avg * (252 / REV_HOLD)
    ew_ann = ew_avg * (252 / REV_HOLD)
    excess_ann = excess * (252 / REV_HOLD)
    
    print(f"{reg:22s} {len(sub):>7d} {g1b_ann:>+10.2%} {ew_ann:>+10.2%} {excess_ann:>+10.2%} {t:>8.2f} {win:>7.1%}")

# ── G-1 vs G-1b 비교 ──────────────────────────────────────────────
print("\n" + "=" * 60)
print("G-1 (급락→반등) vs G-1b (급등→??) 비교")
print("=" * 60)

for reg in sorted(g1_results["regime"].unique()):
    sub_a = g1_results[g1_results["regime"] == reg]
    sub_b = g1b_results[g1b_results["regime"] == reg]
    if len(sub_a) < 20 or len(sub_b) < 20:
        continue
    
    ex_a = (sub_a["g1_ret"].mean() - sub_a["ew_ret"].mean()) * (252 / REV_HOLD)
    ex_b = (sub_b["g1_ret"].mean() - sub_b["ew_ret"].mean()) * (252 / REV_HOLD)
    t_a = calc_tstat(sub_a["g1_ret"], sub_a["ew_ret"])
    t_b = calc_tstat(sub_b["g1_ret"], sub_b["ew_ret"])
    
    print(f"{reg:22s}  급락반전: {ex_a:>+8.2%} (t={t_a:>5.2f})  |  급등후: {ex_b:>+8.2%} (t={t_b:>5.2f})")

print("\n→ G-1b 초과수익이 음수면: 급등 종목 회피 시그널로 활용 가능")
print("→ G-1b 초과수익이 양수면: 모멘텀 효과 (급등 종목이 계속 감)")

G-1b: 급등 종목 매수 (반전 반대편)
총 3172 거래

전체: 초과수익(연율) = +0.70%, t = 0.44

레짐                      trades  Surge avg     EW avg         초과   t-stat      승률
-------------------------------------------------------------------------------------
Bear                       486    +16.96%    +29.65%    -12.68%    -2.05   47.3%
Bull                      2417    +20.07%    +14.97%     +5.10%     3.39   52.3%
Neutral                    269     +6.44%    +21.02%    -14.57%    -2.31   45.4%

G-1 (급락→반등) vs G-1b (급등→??) 비교
Bear                    급락반전:  +36.54% (t= 4.09)  |  급등후:  -12.68% (t=-2.05)
Bull                    급락반전:   +3.87% (t= 2.55)  |  급등후:   +5.10% (t= 3.39)
Neutral                 급락반전:   -0.87% (t=-0.13)  |  급등후:  -14.57% (t=-2.31)

→ G-1b 초과수익이 음수면: 급등 종목 회피 시그널로 활용 가능
→ G-1b 초과수익이 양수면: 모멘텀 효과 (급등 종목이 계속 감)


In [6]:
# =============================================================================
# 셀 5 — G-1 레짐별 성과 분해
# =============================================================================

print("=" * 60)
print("G-1 Reversal — 레짐별 성과")
print("=" * 60)

print(f"\n{'레짐':22s} {'trades':>7s} {'G1 avg':>10s} {'EW avg':>10s} {'초과':>10s} {'t-stat':>8s} {'승률':>7s}")
print("-" * 85)

for reg in sorted(g1_results["regime"].unique()):
    mask = g1_results["regime"] == reg
    sub = g1_results.loc[mask]
    
    if len(sub) < 20:
        print(f"{reg:22s} {len(sub):>7d}   (샘플 부족)")
        continue
    
    g1_avg = sub["g1_ret"].mean()
    ew_avg = sub["ew_ret"].mean()
    excess = g1_avg - ew_avg
    t = calc_tstat(sub["g1_ret"], sub["ew_ret"])
    win = (sub["g1_ret"] > sub["ew_ret"]).mean()
    
    # 연율화 (5일 보유 → 연 50회)
    g1_ann = g1_avg * (252 / REV_HOLD)
    ew_ann = ew_avg * (252 / REV_HOLD)
    excess_ann = excess * (252 / REV_HOLD)
    
    print(f"{reg:22s} {len(sub):>7d} {g1_ann:>+10.2%} {ew_ann:>+10.2%} {excess_ann:>+10.2%} {t:>8.2f} {win:>7.1%}")

G-1 Reversal — 레짐별 성과

레짐                      trades     G1 avg     EW avg         초과   t-stat      승률
-------------------------------------------------------------------------------------
Bear                       486    +66.18%    +29.65%    +36.54%     4.09   57.8%
Bull                      2417    +18.84%    +14.97%     +3.87%     2.55   51.9%
Neutral                    269    +20.15%    +21.02%     -0.87%    -0.13   50.9%


In [7]:
# =============================================================================
# 셀 6 — Lookback × Hold 민감도 분석 (G-1 + G-1b)
# =============================================================================
# 목적: 급락(G-1) + 급등(G-1b) 최적 파라미터 + 과적합 확인
# =============================================================================

print("=" * 60)
print("G-1 / G-1b Lookback × Hold 민감도")
print("=" * 60)

lookbacks = [1, 3, 5, 10, 20]
holds = [1, 3, 5, 10, 20]

sensitivity_results = []

for lb in lookbacks:
    # 시그널 재생성
    past_ret = ret_1d.rolling(window=lb).sum()
    sig_drop = pd.DataFrame(index=past_ret.index, columns=past_ret.columns, dtype=float)
    
    for date in past_ret.index:
        row = past_ret.loc[date].dropna()
        if len(row) < 50:
            continue
        z = (row - row.mean()) / row.std()
        sig_drop.loc[date, z.index] = -z  # 급락=양수
    
    sig_surge = -sig_drop  # 급등=양수
    
    for hd in holds:
        # G-1 (급락 반전)
        bt_drop = backtest_reversal(sig_drop, ret_1d, regime_d, top_n=TOP_N, hold_days=hd)
        # G-1b (급등 후)
        bt_surge = backtest_reversal(sig_surge, ret_1d, regime_d, top_n=TOP_N, hold_days=hd)
        
        if len(bt_drop) < 100 or len(bt_surge) < 100:
            continue
        
        # ── G-1 (급락) ──
        t_all_drop = calc_tstat(bt_drop["g1_ret"], bt_drop["ew_ret"])
        excess_all_drop = (bt_drop["g1_ret"].mean() - bt_drop["ew_ret"].mean()) * (252 / hd)
        
        bear_sub_drop = bt_drop[bt_drop["regime"] == "Bear"]
        if len(bear_sub_drop) >= 10:
            t_bear_drop = calc_tstat(bear_sub_drop["g1_ret"], bear_sub_drop["ew_ret"])
            excess_bear_drop = (bear_sub_drop["g1_ret"].mean() - bear_sub_drop["ew_ret"].mean()) * (252 / hd)
        else:
            t_bear_drop = np.nan
            excess_bear_drop = np.nan
        
        # ── G-1b (급등) ──
        t_all_surge = calc_tstat(bt_surge["g1_ret"], bt_surge["ew_ret"])
        excess_all_surge = (bt_surge["g1_ret"].mean() - bt_surge["ew_ret"].mean()) * (252 / hd)
        
        bear_sub_surge = bt_surge[bt_surge["regime"] == "Bear"]
        if len(bear_sub_surge) >= 10:
            t_bear_surge = calc_tstat(bear_sub_surge["g1_ret"], bear_sub_surge["ew_ret"])
            excess_bear_surge = (bear_sub_surge["g1_ret"].mean() - bear_sub_surge["ew_ret"].mean()) * (252 / hd)
        else:
            t_bear_surge = np.nan
            excess_bear_surge = np.nan
        
        sensitivity_results.append({
            "lookback": lb, "hold": hd,
            # G-1
            "t_all_drop": t_all_drop, "excess_all_drop": excess_all_drop,
            "t_bear_drop": t_bear_drop, "excess_bear_drop": excess_bear_drop,
            # G-1b
            "t_all_surge": t_all_surge, "excess_all_surge": excess_all_surge,
            "t_bear_surge": t_bear_surge, "excess_bear_surge": excess_bear_surge,
            "n_trades": len(bt_drop),
        })

sens_df = pd.DataFrame(sensitivity_results)

# ── G-1 (급락 반전) 피벗 ──────────────────────────────────────────
print("\n[G-1 급락반전 — 전체 t-stat]")
pivot_drop = sens_df.pivot(index="lookback", columns="hold", values="t_all_drop").round(2)
print(pivot_drop)

print("\n[G-1 급락반전 — Bear t-stat]")
pivot_bear_drop = sens_df.pivot(index="lookback", columns="hold", values="t_bear_drop").round(2)
print(pivot_bear_drop)

# ── G-1b (급등 후) 피벗 ──────────────────────────────────────────
print("\n[G-1b 급등후 — 전체 t-stat]")
pivot_surge = sens_df.pivot(index="lookback", columns="hold", values="t_all_surge").round(2)
print(pivot_surge)

print("\n[G-1b 급등후 — Bear t-stat]")
pivot_bear_surge = sens_df.pivot(index="lookback", columns="hold", values="t_bear_surge").round(2)
print(pivot_bear_surge)

# ── 초과수익 피벗 ─────────────────────────────────────────────────
print("\n[G-1b 급등후 — 전체 초과수익(연율)]")
pivot_surge_ex = sens_df.pivot(index="lookback", columns="hold", values="excess_all_surge").round(4)
print((pivot_surge_ex * 100).round(2).astype(str) + "%")

# ── 최적 조합 ─────────────────────────────────────────────────────
best_drop = sens_df.loc[sens_df["t_all_drop"].idxmax()]
print(f"\n★ G-1 전체 최적: lb={best_drop['lookback']:.0f}, hold={best_drop['hold']:.0f} (t={best_drop['t_all_drop']:.2f})")

if sens_df["t_bear_drop"].notna().any():
    best_bear_drop = sens_df.loc[sens_df["t_bear_drop"].idxmax()]
    print(f"★ G-1 Bear 최적: lb={best_bear_drop['lookback']:.0f}, hold={best_bear_drop['hold']:.0f} (t={best_bear_drop['t_bear_drop']:.2f})")

# G-1b: t가 가장 음수인 게 "회피 시그널"로 강한 것
best_surge_neg = sens_df.loc[sens_df["t_all_surge"].idxmin()]
print(f"\n★ G-1b 전체 최악(=회피 최적): lb={best_surge_neg['lookback']:.0f}, hold={best_surge_neg['hold']:.0f} (t={best_surge_neg['t_all_surge']:.2f})")

if sens_df["t_bear_surge"].notna().any():
    best_bear_surge_neg = sens_df.loc[sens_df["t_bear_surge"].idxmin()]
    print(f"★ G-1b Bear 최악: lb={best_bear_surge_neg['lookback']:.0f}, hold={best_bear_surge_neg['hold']:.0f} (t={best_bear_surge_neg['t_bear_surge']:.2f})")

# ── 핵심 해석 ─────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("해석 가이드")
print("=" * 60)
print("G-1b t-stat 음수 → 급등 종목이 이후 EW보다 못함 → 회피 시그널")
print("G-1b t-stat 양수 → 급등 종목이 계속 감 → 모멘텀 효과 (D팩터 영역)")

G-1 / G-1b Lookback × Hold 민감도

[G-1 급락반전 — 전체 t-stat]
hold        1     3     5     10    20
lookback                              
1         1.62  3.55  4.26  5.89  7.21
3         2.50  4.64  5.14  6.37  8.03
5         1.70  3.84  4.50  5.44  6.98
10        2.00  3.33  4.05  4.62  6.38
20        1.23  2.40  2.70  3.32  4.66

[G-1 급락반전 — Bear t-stat]
hold        1     3     5     10    20
lookback                              
1         0.25  1.29  2.24  4.85  6.15
3         1.38  2.86  3.85  5.65  7.68
5         1.43  3.32  4.09  5.23  7.78
10        1.66  2.47  2.73  4.03  6.29
20        0.82  1.62  2.01  3.58  5.53

[G-1b 급등후 — 전체 t-stat]
hold        1     3     5     10    20
lookback                              
1         0.04  0.36  0.34  2.29  5.16
3        -0.51 -0.88 -0.49  2.07  5.53
5        -0.57 -0.42  0.44  2.81  5.51
10        0.19  0.70  1.65  4.33  6.39
20        0.50  1.45  2.31  4.30  5.28

[G-1b 급등후 — Bear t-stat]
hold        1     3     5     10    20
lookback   

In [8]:
# =============================================================================
# 05_G-1.ipynb — 최종 결론 (regime_v4 재검증)
# =============================================================================
"""
# G-1 Short-term Reversal 최종 결론 — regime_v4 재검증

## 가설
  단기 급락 종목이 평균회귀로 반등 (과잉반응 → 복원)
  + G-1b: 급등 종목이 이후 하락하는지 (반대편 검증)

---

## G-1 급락 반전 — 레짐별 성과 (lb=5, hold=5)

  레짐      trades   초과(연율)   t-stat   승률
  Bear       486    +36.54%      4.09    57.8%   ⭐
  Bull      2417     +3.87%      2.55    51.9%   ✅
  Neutral    269     -0.87%     -0.13    50.9%   ❌

## G-1b 급등 후 — 레짐별 성과 (lb=5, hold=5)

  레짐      trades   초과(연율)   t-stat   승률
  Bear       486    -12.68%     -2.05    47.3%   ⭐ 회피 시그널
  Bull      2417     +5.10%     +3.39    52.3%   → 모멘텀 효과
  Neutral    269    -14.57%     -2.31    45.4%   ⭐ 회피 시그널

---

## Lookback × Hold 민감도

### G-1 급락 반전:
  전체 최적: lb=3, hold=20 (t=8.03)
  Bear 최적: lb=5, hold=20 (t=7.78)

### G-1b 급등 후:
  전체 최악(=회피 최적): lb=3, hold=3 (t=-0.88)
  Bear 최악: lb=5, hold=5 (t=-2.05)

  ⚠ G-1b hold=20에서는 전부 양수 → 회피는 단기(hold=3~5)만 유효
  ⚠ G-1b Bear 최악 파라미터 = G-1 Bear 최적 파라미터 (lb=5)
     → 급락 매수 + 급등 회피가 동일 세팅에서 작동

---

## 핵심 발견

1. G-1: v2 폐기 → v4 강력 부활 (전체 t=8.03, Bear t=7.78)
2. G-1b 비대칭: 레짐에 따라 급등의 의미가 반전
   - Bear/Neutral: 급등 = 과잉반응 → 하락 (회피)
   - Bull: 급등 = 추세 확인 → 상승 (모멘텀)
3. hold 길수록 G-1 강함, G-1b는 약해짐 → 시간축 분리

---

## 팩터 매핑 (regime_v4)

  | 레짐     | G-1 역할        | G-1b 역할       | 파라미터        |
  |---------|----------------|----------------|----------------|
  | Bull    | ✅ 보조 매수     | ❌ (모멘텀)     | lb=3, hold=20  |
  | Neutral | ❌ 비활성       | ⚠ 약한 회피     | lb=3, hold=3   |
  | Bear    | ⭐ 메인 매수     | ⭐ 회피 필터     | lb=5, hold=20/5|

"""
print("G-1 regime_v4 재검증 결론 확정")
print()
print("  G-1:  ⭐ 강력 부활 — Bear t=7.78, Bull t=8.03")
print("  G-1b: ⭐ 회피 시그널 — Bear t=-2.05, Neutral t=-2.31")

G-1 regime_v4 재검증 결론 확정

  G-1:  ⭐ 강력 부활 — Bear t=7.78, Bull t=8.03
  G-1b: ⭐ 회피 시그널 — Bear t=-2.05, Neutral t=-2.31


In [11]:
# =============================================================================
# [저장 셀] G-1 + G-1b 시그널 → 06_TheForge용
# =============================================================================

if "winsorize" not in dir():
    def winsorize(s, lower=0.01, upper=0.99):
        lo, hi = s.quantile(lower), s.quantile(upper)
        return s.clip(lo, hi)

from pathlib import Path
import numpy as np
SAVE_DIR = Path(r"C:\QP2\data\interim")

# ── 월간 수익률 (월말 시점 시그널) ──
px_m_local = px_wide.resample("ME").last()
ret_1m_local = px_m_local.pct_change()

# ── G-1 Bull (lookback=3M): 3개월 누적 수익률 ──
cum_3m = ret_1m_local.rolling(3).apply(lambda x: (1+x).prod()-1, raw=True)
g1_bull_long = cum_3m.stack().reset_index()
g1_bull_long.columns = ["date", "ticker", "cum_3m"]
g1_bull_long["date"] = pd.to_datetime(g1_bull_long["date"])
g1_bull_long = g1_bull_long.dropna()
# 급락 = 높은 반전 점수 → 부호 반전
g1_bull_long["g1_bull_z"] = g1_bull_long.groupby("date")["cum_3m"].transform(
    lambda x: -((winsorize(x) - winsorize(x).mean()) / winsorize(x).std())
)

# ── G-1 Bear (lookback=5M): 5개월 누적 수익률 ──
cum_5m = ret_1m_local.rolling(5).apply(lambda x: (1+x).prod()-1, raw=True)
g1_bear_long = cum_5m.stack().reset_index()
g1_bear_long.columns = ["date", "ticker", "cum_5m"]
g1_bear_long["date"] = pd.to_datetime(g1_bear_long["date"])
g1_bear_long = g1_bear_long.dropna()
g1_bear_long["g1_bear_z"] = g1_bear_long.groupby("date")["cum_5m"].transform(
    lambda x: -((winsorize(x) - winsorize(x).mean()) / winsorize(x).std())
)

# ── G-1b: 급등 종목 플래그 (3M 누적 상위 10% = 함정) ──
g1b_long = cum_3m.stack().reset_index()
g1b_long.columns = ["date", "ticker", "cum_3m"]
g1b_long["date"] = pd.to_datetime(g1b_long["date"])
g1b_long = g1b_long.dropna()
g1b_long["g1b_flag"] = g1b_long.groupby("date")["cum_3m"].transform(
    lambda x: (x >= x.quantile(0.90)).astype(int)
)

# ── 병합 ──
g1_signal = g1_bull_long[["date", "ticker", "g1_bull_z"]].merge(
    g1_bear_long[["date", "ticker", "g1_bear_z"]],
    on=["date", "ticker"], how="outer"
).merge(
    g1b_long[["date", "ticker", "g1b_flag"]],
    on=["date", "ticker"], how="outer"
)

g1_signal.to_parquet(SAVE_DIR / "g1_signal.parquet", index=False)
print(f"✅ G-1 시그널 저장: {SAVE_DIR / 'g1_signal.parquet'}")
print(f"   {len(g1_signal)} rows, {g1_signal['ticker'].nunique()} tickers")
print(f"   G-1 Bull non-null: {g1_signal['g1_bull_z'].notna().sum()}")
print(f"   G-1 Bear non-null: {g1_signal['g1_bear_z'].notna().sum()}")
print(f"   G-1b 급등 종목수: {int(g1_signal['g1b_flag'].sum())}")

✅ G-1 시그널 저장: C:\QP2\data\interim\g1_signal.parquet
   72374 rows, 503 tickers
   G-1 Bull non-null: 72374
   G-1 Bear non-null: 71368
   G-1b 급등 종목수: 7307
